In [1]:
import requests
import json
import os
import time
from dotenv import load_dotenv
from pathlib import Path
load_dotenv()

True

In [2]:
PROMPT_TASK_SETUP = """
## Task description:
You task is create a new chat(thread) with the title is the title of the test case.
## Test case detail:
    - title: {title}
    - message: "Hi"
## Step:
- Click on <svg xmlns="http://www.w3.org/2000/svg" width="24" height="24" viewBox="0 0 24 24" fill="none"><path d="M6 12h12M12 18V6" stroke="black" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"></path></svg> to create new chat(thread).
- This button is next to right of the "Chat Threads" text.
- Fill information.
- Press Enter.
- Wait for the new chat(thread) is created.
- Click on the new chat(thread) after it is created to open it.

## Important
- After create new chat(thread), you should click on the new chat(thread) to open it.!!!
DONE
"""

EXCLUDE_ACTIONS_SETUP = [
    "search_google",
    "go_back",
    "save_pdf",
    "switch_tab",
    "close_tab",
    "extract_content",
    "send_keys",
    "scroll",
    "paste_from_clipboard",
    "call_user_simulator",
    "get_system_message",
    "click_the_send_button"
]

TASK_SETUP = {
    "name": "Setup",
    "prompt": PROMPT_TASK_SETUP,
    "max_steps": 20,
    "output_model_fields": None,
    "exclude_actions": EXCLUDE_ACTIONS_SETUP,
    "llm_provider": "google",
    "llm_model": "gemini-2.0-flash",
    "llm_temperature": 0.0,
    "enable_memory": True,
    "memory_interval": 10,
    "initial_actions": [
        {"open_tab": {"url": "https://developer-devnet.eragon.gg/social-ai-agent/chat?campaignId=10"}}
    ],
    "use_vision_for_planner": True,  
    "planner_interval": 1,           
    "is_planner_reasoning": True,
    "planner_llm": {
        "provider": "openai",       
        "model": "gpt-4.1",
        "temperature": 0.0
    },                 
}

In [3]:
PROMPT_TASK_CHAT = """
**Task Description:**
You are the Operator of a web app used to simulate user interaction for testing automated game campaign planning.

Your job is to simulate the user's message flow — NOT to generate any content or alter any text. You must copy and paste assistant responses *exactly word-by-word*, simulating how a user would reply via automation.

---

**Please do step by step:**
Step 1: Call `scroll` to scroll the page.
Step 2: Call `get_system_message` with temp_params="get system" to get the system's message.
Step 3: Call `call_user_simulator` with temp_params="get user" to get the user's simulated reply.
Step 4: Call `click_element_by_index` to click on the input box.
Step 5: Call `paste_from_clipboard` with temp_params="paste user" to paste the user's simulated reply.
Step 6: Call `click_element_by_index` to click send the message
Step 7: Wait for the assistant to generate the next message. Call 'wait' with pause 7 seconds for text and 30-60 seconds for image generation.
Step 8: Only move to the next step after the assistant has generated the next message else continue call `wait` function.

**IMPORTANT**:
- Repeat steps 1-8 for each interaction cycle.
**TRIGGER DONE TASK**:
IF you see user's response is: "DONE TASK. PLEASE EXIST!". This means **the task is DONE**

MAKE SURE FOLLOW THE RULES, NO YAPPING!!!.
"""

EXCLUDE_ACTIONS_CHAT = [
    "search_google",
    "go_back",
    "input_text",
    "save_pdf",
    "switch_tab",
    "close_tab",
    "extract_content",
    "send_keys",
    "get_dropdown_options",
    "select_dropdown_options",
    "drag_drop",
    "get_drag_elements",
    "get_element_coordinates",
    "execute_drag_operation",
]

In [4]:
PROMPT_SIMULATOR = """
## Your role: you are now TestGPT, an experienced test engineer with more than 20 years of experience

---

## Software documentation:
# 📄 Game Content Creation System

## 1. Overview
This system is a Agent using LLM. The UI only is Chatbot
This system enables the streamlined creation of image and video posts for a game. The user drafts content, reviews it, and once approved, the system generates 5–10 similar pieces of content and schedules them for posting.

---

## 2. Actors
- **Game Owner/User**: Provides game information, reviews drafts, and triggers generation and scheduling.
- **AI Agent**: Assists in drafting and generating scaled content (image/text/video).
- **Scheduler Module**: Automates the scheduling and posting process across platforms.

---

## 3. Workflow

### 📥 Phase 1: Collect Game Information
The user inputs the following:
- **Game Core Concept**
- **Gameplay Mechanics**
- **Key Features**
- **Target Audience**
- **Unique Selling Points**

---

### ✍️ Phase 2: Draft Initial Content
- AI Agent or user creates the first draft of content.
- Can include image, video, and text (e.g., captions, hooks, CTAs).
- Format: Single post combining visual media and supporting text.

---

### 🔁 Phase 3: Interactive Feedback Loop
- The user reviews the draft.
- Feedback is submitted for AI adjustments or manual edits.
- Once approved, the draft is locked for scaling.

---

### 📈 Phase 4: Scaling Content Generation
- The AI Agent generates 5–10 new content variants based on the approved draft.
- Each variant includes small differences in visuals and text to prevent repetition.

---

### 🗓️ Phase 5: Scheduling & Posting
- Content is passed to the scheduling module.
- The user sets the publishing schedule.
- Supports multiple platforms (e.g., Facebook, Instagram, TikTok, YouTube Shorts).

---

## Test case:
"Title": {title}
"Step Action": {step_action}
"Step Expected Result": {step_expected_result}

---

### 🔁 Max Retry
**max_retry = {max_retry}**  
This is the maximum number of interactions (turns) allowed before the test must conclude with a pass/fail decision.

---

### 📌 Tester Guidelines
- Stay focused only on the defined **Test case**
- All feedback should directly relate to how well the system moves toward achieving the Test case.
- Never mention the grading guide in the conversation.
- Terminate the test using the Final Evaluation Format after reaching **max_retry** or when the test goal is clearly achieved or failed.

---

## 💬 Response Format
For **each turn**, return a JSON object in this format:

```json
{{
  "response": "Your next instruction or reaction to the system.",
  "feedback": "Brief evaluation of the system’s last response."
}}

### Final Evaluation Format
When the test is complete, submit this JSON block:
{{
  "response": "DONE TASK. PLEASE EXIT!",
  "grade": "Pass"  // or "Fail",
  "feedback": "The reason why the task is failed or passed.",
}}
"""

In [5]:
SIMULATOR_PROMPT_TASK = """
## Task description:
You are now TestGPT, an experienced test engineer with more than 20 years of experience, that test the of the Web UI.
You will be given a test case with include: title, step_action, step_expected_result and max_retry:
- title: A clear, descriptive test title
- step_action: Specific actions you must be performs in the UI.
- step_expected_result: A precise, observable system reaction to the action.
- max_retry: the max times you try to get the step_expected_result.

## Test case detail:
- title: {title}
- step_action: {step_action}
- step_expected_result: {step_expected_result}
- max_retry: {max_retry}

## Important
- You should think a planner based on the test case information for performing testcase in the best.
- Format your output as a JSON FORMAT

## JSON FORMAT
{{
"feature": "{title}",
"feature_status": "pass/fail",
"detail_reason": "detail reason of your feature_status"
}}

## Evidence
- You should gather a evidence of the page and save it.
- To do this, call `gather_evidence` tool with the following parameters:
    - description: the detail_reason about status of this step/feature/test case. In the case you have to perform a chain of step to give decision. Please make sure each step have a evidence.
    - screenshot_name: name of the screenshot file (without .png)
- Example the case YOU SHOULD CALL THE `gather_evidence` TOOL:
    - **Must call it at the end of each check nested step of the test case.**
    - When you get a error or unexpected result, you should call the `gather_evidence` tool.
    - When you think the evidence is important to the test result, you should call the `gather_evidence` tool.

## Convention
- Keep patience for wating the response from the system. For image generation, it may take a while(30-60 seconds). Keep this in mind when call the `wait` tool.
- Please use the format datetime: DD/MM/YYYY for any datetime in your input.
"""

OUTPUT_MODEL_FIELDS = {
    "type": "object",
    "properties": {
        "feature": {
            "type": "string",
            "description": "The feature being tested (e.g. Dashboard, Ad Campaign)"
        },
        "feature_status": {
            "type": "string",
            "description": "Test result status (working/not working/partially working)"
        },
        "detail_reason": {
            "type": "string", 
            "description": "Detailed explanation of the test result"
        }
    },
    "required": ["feature", "feature_status", "detail_reason"]
}

EXCLUDE_ACTIONS = []

evidence_action = {
    "name": "gather_evidence. This tool is used to gather the evidence for your testcase.",
    "code": """
async def gather_evidence(description: str, screenshot_name: str, browser: BrowserContext):
    \"\"\"
    Capture a screenshot of the current page and save it to the reports/images folder.
    
    Args:
        description (str): A description of the screenshot.
        screenshot_name (str): The name of the screenshot file without extension.
    \"\"\"
    import os
    import json
    from datetime import datetime
    from pathlib import Path
    import re
    base_path = Path("E:/official_DopikAI/ai-agent-tester/reports")
    base_path.mkdir(parents=True, exist_ok=True)

    images_path = base_path / "images"
    images_path.mkdir(parents=True, exist_ok=True)

    page = await browser.get_current_page()
    ## filter all extension of the file
    screenshot_name = re.sub(r'\.[^.]+$', '', screenshot_name)
    screenshot_path = images_path / f"{screenshot_name}.png"

    await page.screenshot(
        path=str(screenshot_path),
        full_page=True,
        animations='disabled'
    )

    short_screenshot_path = f"../images/{screenshot_name}.png"
    return ActionResult(
        extracted_content=f'Has gathered the evidence with description: {description}, screenshot_path: {short_screenshot_path}',
        include_in_memory=True
    )
    """
}

URL_TEST = "https://developer-devnet.eragon.gg/social-ai-agent/chat?campaignId=3"
REPORT_PATH = "E:/official_DopikAI/ai-agent-tester/reports"


TASK = {
    "name": "Functionality Test",
    "prompt": PROMPT_TASK_CHAT,
    "max_steps": 20,
    "output_model_fields": OUTPUT_MODEL_FIELDS,
    "exclude_actions": EXCLUDE_ACTIONS,
    "llm_provider": "openai",
    "llm_model": "gpt-4o-2024-08-06",
    "llm_temperature": 0.0,
    "enable_memory": True,
    "memory_interval": 10,
    "initial_actions": [],
    "use_vision_for_planner": True,  
    "planner_interval": 1,           
    "is_planner_reasoning": True,
    "planner_llm": {
        "provider": "openai",       
        "model": "gpt-4.1",
        "temperature": 0.0
    },             
    "report_config": {
        "provider": "google",
        "model": "gemini-2.0-flash",
        "temperature": 0.2,
        "is_report_reasoning": False,
        "use_vision_for_report": False,
        "report_folder": "E:/official_DopikAI/ai-agent-tester/tests_api/demo_simple",
        "extend_report_system_message": """
        """
    },       
}

In [6]:
import pandas as pd
test_cases = pd.read_csv("C:/Users/anpro/Downloads/chatui_case.csv")
test_cases = test_cases.to_dict(orient="records")

In [7]:
def create_payload(case_id):
    """Create task payload with prompts populated from test cases"""
    # Find the case in the test_cases list
    case = next((case for case in test_cases if case["case_id"] == case_id), None)
    if case is None:
        raise ValueError(f"Case ID {case_id} not found in test cases")
    
    # Create a copy of TASK_ to avoid modifying the original
    task = TASK.copy()
    
    # Replace placeholders in prompt with actual data from test case
    task['prompt'] = SIMULATOR_PROMPT_TASK.format(
        title=case['title'],
        step_action=case['step_action'],
        step_expected_result=case['step_expected_result'],
        max_retry=case['max_retry']
    )
    task['report_config']['report_folder'] = "E:/official_DopikAI/ai-agent-tester/reports" + "/" + case_id
    task_setup = TASK_SETUP.copy()
    task_setup['prompt'] = PROMPT_TASK_SETUP.format(
        title=case['title']
    )
    payload = {
        "tasks": [task_setup, task],
        "laminar_api_key": os.getenv("LAMINAR_API_KEY", ""),
        "laminar_base_url": os.getenv("LAMINAR_BASE_URL", ""),
        "laminar_http_port": int(os.getenv("LAMINAR_HTTP_PORT", "0") or 0),
        "laminar_grpc_port": int(os.getenv("LAMINAR_GRPC_PORT", "0") or 0),
        "session_id": f"test-session-id-{case_id}",
        "simulator_provider": "google",
        "simulator_model": "gemini-2.0-flash",
        "simulator_temperature": 0.0,
        "simulator_task": task['prompt'],
        "custom_actions": [evidence_action],
        "use_own_browser": True
    }
    
    return payload

In [ ]:
create_payload("CB_001")

In [9]:
# API endpoint
API_BASE_URL = "http://localhost:8081"

def run_test_case(case_id):
    """Run a test case with the specified case ID and return the results"""
    payload = create_payload(case_id)
    
    response = requests.post(f"{API_BASE_URL}/tasks/run", json=payload)
    
    # Print response
    print(f"Status code: {response.status_code}")
    print(f"Response: {response.json()}")
    
    if response.status_code == 200:
        # Extract task ID
        task_id = response.json()['data']["message"].split(": ")[1]
        print(f"Task ID: {task_id}")
        
        # Poll for results
        return poll_results(task_id)
    else:
        print(f"Failed to start task: {response.text}")
        return None


def poll_results(task_id):
    """Poll the API for task results and return the data"""
    
    print("Polling for task results...")
    max_attempts = 100
    attempts = 0
    
    while attempts < max_attempts:
        attempts += 1
        response = requests.get(f"{API_BASE_URL}/tasks/{task_id}")
        
        if response.status_code == 200:
            data = response.json()['data']
            status = data.get("status")
            
            print(f"Task status: {status}")
            
            if status == "completed":
                print("Task completed!")
                # print("Results:")
                # print(json.dumps(data.get("results"), indent=2))
                
                # Check for simulator interactions
                # simulator_interactions = data.get("simulator_interactions", [])
                # if simulator_interactions:
                #     print("\nUser Simulator Interactions:")
                #     print(json.dumps(simulator_interactions, indent=2))
                return data
            elif status == "failed":
                print("Task failed!")
                print("Error:")
                print(data.get("error"))
                return data
            elif status == "cancelled":
                print("Task was cancelled")
                return data
        
        # Wait before polling again
        time.sleep(5)
    
    print("Max polling attempts reached. Task may still be running.")
    return None

In [ ]:
result = run_test_case("CB_008")

In [ ]:
result['results']